In [8]:
def mru(reference, frame_size):
    frames = [None] * frame_size       # F1..FN
    last_used = {}                     # page -> last access time
    hits = faults = 0
    insert_seq, replace_seq, rows = [], [], []

    for t, page in enumerate(reference):
        if page in frames:
            hits += 1
            status, action = "HIT", "-"
        else:
            faults += 1
            status = "MISS (FAULT)"
            if None in frames:                         # empty slot ache
                frames[frames.index(None)] = page
                insert_seq.append(page)
                action = f"Insert {page}"
            else:                                      # MRU page er slot e replace
                slot = max(range(frame_size), key=lambda i: last_used[frames[i]])
                victim = frames[slot]
                frames[slot] = page
                replace_seq.append((victim, page))
                action = f"Replace {victim} -> {page}"
        last_used[page] = t                            # access time update
        rows.append((page, list(frames), status, action))

    total = len(reference)
    return dict(rows=rows, hits=hits, faults=faults,
                hit_ratio=hits/total, fault_ratio=faults/total,
                insert_seq=insert_seq, replace_seq=replace_seq, total=total)


def print_result(reference, frame_size):
    r = mru(reference, frame_size)
    print(f"Reference String : {' '.join(map(str, reference))}")
    print(f"Frame Size       : {frame_size}\n")

    head = f"{'Page':<6}" + "".join(f"{'F'+str(i+1):<5}" for i in range(frame_size)) \
           + f"{'Status':<15}Action"
    print(head)
    print("-" * (len(head) + 12))
    for page, fr, status, action in r["rows"]:
        cells = "".join(f"{('-' if x is None else x)!s:<5}" for x in fr)
        print(f"{page:<6}{cells}{status:<15}{action}")

    print()
    print(f"Total References : {r['total']}")
    print(f"Hits             : {r['hits']}")
    print(f"Misses (Faults)  : {r['faults']}")
    print(f"Hit Ratio        : {r['hits']}/{r['total']} = {r['hit_ratio']:.4f} ({r['hit_ratio']*100:.2f}%)")
    print(f"Fault Ratio      : {r['faults']}/{r['total']} = {r['fault_ratio']:.4f} ({r['fault_ratio']*100:.2f}%)")
    print(f"Insert Sequence  : {' -> '.join(map(str, r['insert_seq']))}")
    rep = ", ".join(f"{v}->{n}" for v, n in r["replace_seq"]) or "None"
    print(f"Replace Sequence : {rep}   (old page -> new page)")


# Whiteboard example
reference = [1, 2, 3, 4, 5, 1, 2, 3, 1]
frame_size = 3
print_result(reference, frame_size)

Reference String : 1 2 3 4 5 1 2 3 1
Frame Size       : 3

Page  F1   F2   F3   Status         Action
------------------------------------------------------
1     1    -    -    MISS (FAULT)   Insert 1
2     1    2    -    MISS (FAULT)   Insert 2
3     1    2    3    MISS (FAULT)   Insert 3
4     1    2    4    MISS (FAULT)   Replace 3 -> 4
5     1    2    5    MISS (FAULT)   Replace 4 -> 5
1     1    2    5    HIT            -
2     1    2    5    HIT            -
3     1    3    5    MISS (FAULT)   Replace 2 -> 3
1     1    3    5    HIT            -

Total References : 9
Hits             : 3
Misses (Faults)  : 6
Hit Ratio        : 3/9 = 0.3333 (33.33%)
Fault Ratio      : 6/9 = 0.6667 (66.67%)
Insert Sequence  : 1 -> 2 -> 3
Replace Sequence : 3->4, 4->5, 2->3   (old page -> new page)
